In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.simplefilter('ignore')

SEED = 30
K = 10
NUM_CLASSES = 7

In [2]:
df_test = pd.read_csv("/kaggle/input/playground-series-s5e6/test.csv")
df_train = pd.read_csv("/kaggle/input/playground-series-s5e6/train.csv")
df_original = pd.read_csv("/kaggle/input/fertilizer-prediction/Fertilizer Prediction.csv")

## Feature Engineering

In [3]:
le = LabelEncoder()
le.fit(df_train['Fertilizer Name'])

def make_features(df, test=False, original=False):
    df_temp = df.copy()
    if not original:
        df_temp.drop(columns=['id'], inplace=True)
    cat_cols = df_temp.select_dtypes(include=['object']).columns
    df_temp[cat_cols] = df_temp[cat_cols].astype('category')

    # adding binning of numerical features
    numerical_features = [col for col in df_temp.select_dtypes(include=['int64', 'float64']).columns]
    for col in numerical_features:
        df_temp[f'{col}_Binned'] = df_temp[col].astype(str).astype('category')

    if not test:
        df_temp['Fertilizer Name'] = le.transform(df_temp['Fertilizer Name'])
    
    return df_temp

In [4]:
def mapk(actual, predicted, k=3):
    total_score = 0.0
    actual = le.inverse_transform(actual)
    for a, p in zip(actual, predicted):
        if a in p[:k]:
            index = p.index(a)
            total_score += 1.0 / (index + 1)
    return total_score / len(actual)

In [5]:
df_train1 = make_features(df_train)
df_original1 = make_features(df_original, original=True)
df_test1 = make_features(df_test, test=True)

In [6]:
initial_params = {
    "tree_method": "gpu_hist",
    "predictor": "gpu_predictor",
    'seed': SEED,
    'enable_categorical': True,
    'early_stopping_rounds': 100
}

In [7]:
X = df_train1.drop(columns=['Fertilizer Name'])
y = df_train1['Fertilizer Name']

X_original = df_original1.drop(columns=['Fertilizer Name'])
y_original = df_original1['Fertilizer Name']

X_original_copy = X_original.copy()
y_original_copy = y_original.copy()

In [8]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=SEED)

In [9]:
model_params = [
    # from https://www.kaggle.com/code/hahahaj/single-xgb
    [{
        'objective': 'multi:softprob',  
        'num_class': 7, 
        'max_depth': 7,
        'learning_rate': 0.03,
        'subsample': 0.8,
        'max_bin': 128,
        'colsample_bytree': 0.3, 
        'colsample_bylevel': 1,  
        'colsample_bynode': 1,  
        'tree_method': 'hist',  
        'random_state': SEED,
        'eval_metric': 'mlogloss',
        'device': "cuda",
        'enable_categorical':True,
        'n_estimators':10000,
        'early_stopping_rounds':50,
    }, 7], # weight of original

    # my own hyperparams from tuning without using original data
    [{'learning_rate': 0.03,
       'max_depth': 11,
       'subsample': 0.8,
       'colsample_bytree': 0.30000000000000004,
       'max_bin': 1551,
       'min_child_weight': 3,
       'gamma': 0.0,
       'lambda': 0.0013312723042592412,
       'alpha': 0.6136573473631746,
       'max_delta_step': 8,
       'n_estimators': 3000,
       'enable_categorical': True,
       'early_stopping_rounds': 100,
       'random_state': SEED,
       'device': "cuda"
     }, 7],

    # my own hyperparams from tuning optuna with original data
     [{'learning_rate': 0.05,
       'max_depth': 7,
       'subsample': 0.8,
       'colsample_bytree': 0.4,
       'max_bin': 1323,
       'min_child_weight': 4,
       'gamma': 0.03,
       'lambda': 0.00953514350973168,
       'alpha': 0.6191568184269528,
       'max_delta_step': 3,
       "n_estimators": 3000,
       "enable_categorical": True,
       'early_stopping_rounds': 100,
       'random_state': SEED,
       'device': "cuda"
       }, 3]
    

]

In [10]:
# Setting Objects Containing Models and Iteration Parameters
kf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)

base_models = [XGBClassifier(**(params[0])) for params in model_params]
original_iterations = [params[1] for params in model_params]

N_MODELS = len(base_models)

In [11]:
# # Making OOF Predictions
# oof_train = np.zeros((len(X_train), N_MODELS * NUM_CLASSES))
# val_preds = np.zeros((len(X_val), N_MODELS * NUM_CLASSES))

# for m_idx, model in enumerate(base_models):
#     val_fold_preds = []
#     scores = []
#     fold = 0

#     for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train), 1):
#         X_original_fold = pd.concat([X_original_copy] * original_iterations[m_idx])
#         y_original_fold = pd.concat([y_original_copy] * original_iterations[m_idx])

#         X_train_fold = pd.concat([X_train.iloc[train_idx], X_original_fold]).reset_index(drop=True)
#         y_train_fold = pd.concat([y_train.iloc[train_idx], y_original_fold]).reset_index(drop=True)
#         X_val_fold = X_train.iloc[val_idx]
#         y_val_fold = y_train.iloc[val_idx]


#         model.fit(
#             X_train_fold,
#             y_train_fold,
#             eval_set=[(X_val_fold, y_val_fold)],
#             verbose=250,
#         )

#         # Predict on validation fold and store in OOF matrix
#         probas = model.predict_proba(X_val_fold)
#         oof_train[val_idx, m_idx*NUM_CLASSES:(m_idx+1)*NUM_CLASSES] = probas

#         # Predict on test data and save for averaging
#         val_pred = model.predict_proba(X_val)
#         val_fold_preds.append(val_pred)

#         val_fold_pred = model.predict_proba(X_val_fold)
#         val_fold_pred = np.argsort(val_fold_pred, axis=1)[:, -3:][:, ::-1]
#         val_fold_pred = [[le.classes_[j] for j in row] for row in val_fold_pred]
#         score = mapk(y_val_fold, val_fold_pred)
#         scores.append(score)

#         print(f"XGB Model {m_idx+1} fold {fold} val score: {score}")

#     avg_score = np.mean(scores)
#     print(f"========== XGB Model {m_idx+1} Average Val Score: {avg_score} ==========")
#     print('\n')
#     val_preds[:, m_idx*NUM_CLASSES:(m_idx+1)*NUM_CLASSES] = np.mean(val_fold_preds, axis=0)

In [12]:
# # saving so we don't have to run oof loop every time
# np.save('oof_train.npy', oof_train)
# np.save('val_preds.npy', val_preds)

### Loading OOF Train and Validation

In [13]:
# Using Data From Earlier Run
oof_train = np.load('/kaggle/input/fertilizers-oof-train-data/oof_train.npy')
val_preds = np.load('/kaggle/input/fertilizers-oof-train-data/val_preds.npy')

## Meta Model

### Baseline Logistic Regression Meta Model

In [14]:
from sklearn.linear_model import LogisticRegression

In [15]:
lr_meta_model = LogisticRegression(max_iter=1000, random_state=SEED)
lr_meta_model.fit(oof_train, y_train)
meta_val_preds = lr_meta_model.predict_proba(val_preds)
meta_val_preds = np.argsort(meta_val_preds, axis=1)[:, -3:][:, ::-1]
meta_val_preds = [[le.classes_[j] for j in row] for row in meta_val_preds]
score = mapk(y_val, meta_val_preds)
print(f"Logistic Regression Meta Model Val Score: {score}")

Logistic Regression Meta Model Val Score: 0.3790577777777219


### LightGBM Meta Model

In [16]:
import lightgbm as lgb

In [17]:
lgb_meta_model = lgb.LGBMClassifier(
    device='gpu'
)
lgb_meta_model.fit(oof_train, y_train)
meta_val_preds = lgb_meta_model.predict_proba(val_preds)
meta_val_preds = np.argsort(meta_val_preds, axis=1)[:, -3:][:, ::-1]
meta_val_preds = [[le.classes_[j] for j in row] for row in meta_val_preds]
score = mapk(y_val, meta_val_preds)
print(f"LGB Meta Model Val Score: {score}")

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 5355
[LightGBM] [Info] Number of data points in the train set: 675000, number of used features: 21
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 21 dense feature groups (15.45 MB) transferred to GPU in 0.019123 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -1.885084
[LightGBM] [Info] Start training from score -1.880343
[LightGBM] [Info] Start training from score -1.897150
[LightGBM] [Info] Start training from score -1.911935
[LightGBM] [Info] Start training from score -1.909163
[LightGBM] [Info] Start training from score -2.068280
[LightGBM] [Info] Start training from score -2.093549
LGB Meta Model Val Score: 0.3791533333332778


### XGBoost Meta Model

In [18]:
xgb_meta_model = XGBClassifier(
    device='cuda'
)
xgb_meta_model.fit(oof_train, y_train)
meta_val_preds = xgb_meta_model.predict_proba(val_preds)
meta_val_preds = np.argsort(meta_val_preds, axis=1)[:, -3:][:, ::-1]
meta_val_preds = [[le.classes_[j] for j in row] for row in meta_val_preds]
score = mapk(y_val, meta_val_preds)
print(f"XGB Meta Model Val Score: {score}")

XGB Meta Model Val Score: 0.3763511111110565


### Neural Network Meta Model

**This is not used**

In [19]:
import torch
from torch import nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

In [20]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [21]:
class mapk_loss_criterion(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, y_pred, y_actual, k=3):
        topk_preds = torch.topk(y_pred, k, dim=1).indices
        y_actual = y_actual.view(-1, 1).expand_as(topk_preds)
    
        correct = (topk_preds == y_actual).float()

        precision_at_k = correct / (torch.arange(1, k+1, device=y_pred.device).float())
        ap = precision_at_k * correct
    
        return ap.sum(dim=1).mean().item()

class MetaNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(N_MODELS * NUM_CLASSES, 32),
            nn.ReLU(),
            nn.Linear(32, 7)
        )

    def forward(self, X):
        return self.layers(X)

In [22]:
# LEARNING_RATE = 1e-3
# BATCH=64
# EPOCHS=150

# mapk_loss = mapk_loss_criterion()

In [23]:
# oof_train = torch.tensor(oof_train, dtype=torch.float32).to(device)
# y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).to(device)
# val_preds = torch.tensor(val_preds, dtype=torch.float32).to(device)
# y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).to(device).squeeze().long()

# dataset = TensorDataset(oof_train, y_train_tensor)
# loader = DataLoader(dataset, batch_size = BATCH)

In [24]:
# all_train_ce_loss = []
# all_train_mapk_loss = []
# all_val_ce_loss = []
# all_val_mapk_loss = []

# torch.manual_seed(SEED)

# meta_model = MetaNN().to(device)

# criterion = nn.CrossEntropyLoss() # not using map@3 since not differentiable
# optimizer = optim.Adam(meta_model.parameters(), lr=LEARNING_RATE)

# for epoch in tqdm(range(EPOCHS)):
#     meta_model.train()
#     epoch_ce_loss = 0 # cross entropy loss
#     epoch_mapk_loss = 0
#     for X_batch, y_batch in loader:
#         X_batch = X_batch.to(device)
#         y_batch = y_batch.to(device).squeeze().long()
#         optimizer.zero_grad()
    
#         logits = meta_model(X_batch)
#         loss = criterion(logits, y_batch)
#         epoch_ce_loss += loss.item()

#         y_probs = torch.softmax(logits, dim=1)
#         batch_mapk_loss = mapk_loss(y_probs, y_batch)
#         epoch_mapk_loss += batch_mapk_loss
        
#         loss.backward()
#         optimizer.step()
    
#     # validation
#     epoch_ce_loss /= len(loader)
#     epoch_mapk_loss /= len(loader)
#     all_train_ce_loss.append(epoch_ce_loss)
#     all_train_mapk_loss.append(epoch_mapk_loss)
#     meta_model.eval()
#     with torch.no_grad():
#         logits = meta_model(val_preds)
#         val_ce_loss = criterion(logits, y_val_tensor)
#         all_val_ce_loss.append(val_ce_loss.item())

#         y_probs = torch.softmax(logits, dim=1)
#         val_mapk_loss = mapk_loss(y_probs, y_val_tensor)
#         all_val_mapk_loss.append(val_mapk_loss)
        

#     if (epoch + 1) % 25 == 0 or epoch == 0:
#         print(f"========== EPOCH :{epoch} ==========")
#         print(f"Training Cross Entropy Loss: {epoch_ce_loss}")
#         print(f"Training MAP@3 Loss: {epoch_mapk_loss}")
#         print(f"Validation Cross Entropy Loss: {val_ce_loss}")
#         print(f"Validation MAP@3 Loss: {val_mapk_loss}")

In [25]:
# torch.save(meta_model, 'meta_model.pth')

In [26]:
# # Meta Model Training Plot
# fig, axes = plt.subplots(1, 2, figsize=(12, 6))
# axes = axes.flatten()

# axes[0].plot(all_train_ce_loss, label='Training Loss')
# axes[0].plot(all_val_ce_loss, label='Validation Loss')
# axes[0].set_xlabel('Epoch')
# axes[0].set_ylabel('Loss')
# axes[0].set_title('Cross Entropy Loss')
# axes[0].legend()
# axes[0].grid(True)

# axes[1].plot(all_train_mapk_loss, label='Training Loss')
# axes[1].plot(all_val_mapk_loss, label='Validation Loss')
# axes[1].set_xlabel('Epoch')
# axes[1].set_ylabel('Loss')
# axes[1].set_title('MAP@3 Loss')
# axes[1].legend()
# axes[1].grid(True)


# plt.tight_layout()
# plt.show()

## Submission

In [27]:
df_test1 = make_features(df_test, test=True)

In [28]:
# Store models and predictions
test_preds = np.zeros((len(df_test1), N_MODELS * NUM_CLASSES))

for i in range(N_MODELS):
    model = base_models[i]

    # Extend training data with original data replicated as needed
    X_original_repeated = pd.concat([X_original_copy] * original_iterations[i])
    y_original_repeated = pd.concat([y_original_copy] * original_iterations[i])
    X_train_model = pd.concat([X_train, X_original_repeated]).reset_index(drop=True)
    y_train_model = pd.concat([y_train, y_original_repeated]).reset_index(drop=True)

    # Train the model
    model.fit(X_train_model, y_train_model, eval_set=[(X_val, y_val)], verbose=250)

    # Predict probabilities and store in the correct slice
    test_preds[:, i * NUM_CLASSES:(i + 1) * NUM_CLASSES] = model.predict_proba(df_test1)

[0]	validation_0-mlogloss:1.94564
[250]	validation_0-mlogloss:1.91717
[500]	validation_0-mlogloss:1.90499
[750]	validation_0-mlogloss:1.89683
[1000]	validation_0-mlogloss:1.89092
[1250]	validation_0-mlogloss:1.88683
[1500]	validation_0-mlogloss:1.88418
[1750]	validation_0-mlogloss:1.88254
[2000]	validation_0-mlogloss:1.88177
[2249]	validation_0-mlogloss:1.88152
[0]	validation_0-mlogloss:1.94546
[250]	validation_0-mlogloss:1.89785
[500]	validation_0-mlogloss:1.88562
[736]	validation_0-mlogloss:1.88427
[0]	validation_0-mlogloss:1.94542
[250]	validation_0-mlogloss:1.90862
[500]	validation_0-mlogloss:1.89558
[750]	validation_0-mlogloss:1.88817
[1000]	validation_0-mlogloss:1.88430
[1250]	validation_0-mlogloss:1.88274
[1461]	validation_0-mlogloss:1.88261


In [29]:
# # Meta Model Test Predictions for Neural Network
# test_preds_tensor = torch.tensor(test_preds, dtype=torch.float32).to(device)
# logits = meta_model(test_preds_tensor)
# y_test_pred = torch.softmax(logits, dim=1)
# y_test_pred = y_test_pred.cpu().detach().numpy()
# y_test_pred = np.argsort(y_test_pred, axis=1)[:, -3:][:, ::-1]
# y_test_pred = [[le.classes_[j] for j in row] for row in y_test_pred]
# y_test_pred = [' '.join(row) for row in y_test_pred]

# submission = pd.read_csv("/kaggle/input/playground-series-s5e6/sample_submission.csv")
# submission['Fertilizer Name'] = y_test_pred
# submission.to_csv('submission.csv', index=False)
# submission.head()

In [30]:
# For anything that isn't the neural network
y_test_pred = lr_meta_model.predict_proba(test_preds)
y_test_pred = np.argsort(y_test_pred, axis=1)[:, -3:][:, ::-1]
y_test_pred = [[le.classes_[j] for j in row] for row in y_test_pred]
y_test_pred = [' '.join(row) for row in y_test_pred]

submission = pd.read_csv("/kaggle/input/playground-series-s5e6/sample_submission.csv")
submission['Fertilizer Name'] = y_test_pred
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Fertilizer Name
0,750000,10-26-26 20-20 14-35-14
1,750001,17-17-17 10-26-26 28-28
2,750002,20-20 Urea DAP
3,750003,14-35-14 Urea 10-26-26
4,750004,Urea 20-20 10-26-26
